SetUp & Data Loading

In [ ]:
from huggingface_hub import login
login()

In [ ]:
!pip install "datasets<3.0.0" librosa scikit-learn hmmlearn jiwer evaluate huggingface_hub

  Using cached datasets-2.21.0-py3-none-any.whl.metadata (21 kB)
  Using cached hmmlearn-0.3.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 71.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. T

In [ ]:
from datasets import load_dataset
from transformers import WhisperProcessor
from datasets import Audio
from transformers import WhisperFeatureExtractor
from transformers import WhisperTokenizer
from transformers import WhisperForConditionalGeneration
from transformers import Seq2SeqTrainingArguments
from transformers import Seq2SeqTrainer
from transformers import pipeline

import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

import evaluate

In [ ]:
sindhi_asr = load_dataset("google/fleurs", "sd_in", trust_remote_code=True)  # for Sindhi
# to download all data for multi-lingual fine-tuning uncomment following line
# fleurs_asr = load_dataset("google/fleurs", "all")

# see structure
print(sindhi_asr)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 3443
    })
    validation: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 426
    })
    test: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 980
    })
})


In [ ]:
# Alias so the rest of the code works unchanged
dataset = sindhi_asr
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

In [ ]:
!pip install librosa

In [ ]:
import librosa
import numpy as np

 Feature Extraction (MFCCs — replaces Whisper's log-mel spectrogram)

In [ ]:
def extract_mfcc(audio_array, sr=16000, n_mfcc=13, n_fft=512, hop_length=160):
    """
    Extract MFCC + delta + delta-delta features.
    Shape: (T, 39) — analogous to Whisper's (80, 3000) log-mel input.
    """
    mfcc = librosa.feature.mfcc(
        y=audio_array.astype(np.float32),
        sr=sr,
        n_mfcc=n_mfcc,
        n_fft=n_fft,
        hop_length=hop_length
    )  # (13, T)

    delta = librosa.feature.delta(mfcc)         # 1st derivative
    delta2 = librosa.feature.delta(mfcc, order=2)  # 2nd derivative

    features = np.vstack([mfcc, delta, delta2]).T  # (T, 39)
    return features


# Build feature matrix for frame-level tasks (e.g., phoneme classification)
def prepare_features(split, max_samples=None):
    features_list, labels_list = [], []
    data = dataset[split]
    if max_samples:
        data = data.select(range(max_samples))

    for sample in data:
        feats = extract_mfcc(sample["audio"]["array"])
        features_list.append(feats)
        labels_list.append(sample["raw_transcription"])

    return features_list, labels_list

train_feats, train_labels = prepare_features("train")
test_feats, test_labels   = prepare_features("test")

In [ ]:
!pip install hmmlearn

 Acoustic Model — GMM-HMM (the classical standard)

In [ ]:
from hmmlearn import hmm
from sklearn.preprocessing import LabelEncoder
import numpy as np
import time

MAX_SEGMENTS_PER_CHAR = 500  # cap to prevent common chars taking forever

def build_char_hmm_models(transcriptions, features_list, n_components=2, n_mix=2, n_iter=5):
    """
    Train one GMM-HMM per character (simplified monophone model).
    """
    print("Step 1/3: Collecting frames per character...")
    char_frames = {}

    for idx, (feats, text) in enumerate(zip(features_list, transcriptions)):
        if idx % 100 == 0:
            print(f"  Processed {idx}/{len(features_list)} utterances", end="\r")

        T = len(feats)
        chars = list(text)
        if not chars:
            continue
        frames_per_char = max(1, T // len(chars))

        for i, ch in enumerate(chars):
            start = i * frames_per_char
            end   = min((i + 1) * frames_per_char, T)
            segment = feats[start:end]
            if len(segment) > 0:
                char_frames.setdefault(ch, []).append(segment)

    print(f"\n  Done. Found {len(char_frames)} unique characters.")
    print(f"  Characters: {sorted(char_frames.keys())}")

    # Show segment counts so you can see which chars are heavy
    print("\n  Segment counts per character:")
    for ch, segs in sorted(char_frames.items(), key=lambda x: -len(x[1])):
        capped = "  ← will be capped" if len(segs) > MAX_SEGMENTS_PER_CHAR else ""
        print(f"    '{ch}': {len(segs)} segments{capped}")

    print("\nStep 2/3: Training GMM-HMM per character...")
    models = {}
    total = len(char_frames)
    start_time = time.time()

    for i, (char, segments) in enumerate(char_frames.items()):
        if len(segments) < 2:
            print(f"  [{i+1}/{total}] Skipping '{char}' — only {len(segments)} segment(s)")
            continue

        # Cap segments to avoid huge chars taking forever
        original_count = len(segments)
        if len(segments) > MAX_SEGMENTS_PER_CHAR:
            segments = segments[:MAX_SEGMENTS_PER_CHAR]

        X = np.vstack(segments)
        lengths = [len(s) for s in segments]

        elapsed = time.time() - start_time
        avg_per_char = elapsed / max(i, 1)
        remaining = avg_per_char * (total - i)

        capped_note = f" (capped from {original_count})" if original_count > MAX_SEGMENTS_PER_CHAR else ""
        print(f"  [{i+1}/{total}] Training '{char}' | {len(segments)} segs{capped_note} | "
              f"{X.shape[0]} frames | ETA: {remaining:.0f}s", end="\r")

        model = hmm.GMMHMM(
            n_components=n_components,
            n_mix=n_mix,
            covariance_type="diag",
            n_iter=n_iter,
            verbose=False
        )
        try:
            model.fit(X, lengths)
            models[char] = model
            print(f"  [{i+1}/{total}] ✓ '{char}' | {len(segments)} segs{capped_note} | "
                  f"{X.shape[0]} frames | ETA: {remaining:.0f}s     ")
        except Exception as e:
            print(f"  [{i+1}/{total}] ✗ Skipped '{char}': {e}     ")

    total_time = time.time() - start_time
    print(f"\nStep 2 done in {total_time:.1f}s. Trained {len(models)}/{total} models.")
    print("\nStep 3/3: Models ready for decoding.")
    return models


# Calculate average frames per character from training data
avg_chars = np.mean([len(t) for t in train_labels])
avg_frames = np.mean([len(f) for f in train_feats])
frames_per_char = int(avg_frames / avg_chars)
print(f"Estimated frames per character: {frames_per_char}")
print(f"Average utterance length: {avg_frames:.0f} frames, {avg_chars:.0f} chars")


def decode_with_hmm(features, models, frames_per_char=frames_per_char):
    """
    Greedy decode: split features into chunks, score each character model.
    Uses data-driven frames_per_char instead of assuming 10 chars per utterance.
    """
    chars = list(models.keys())
    T = len(features)
    chunk_size = max(1, frames_per_char)  # ← now data-driven instead of T // 10

    predicted = []
    for start in range(0, T, chunk_size):
        chunk = features[start:start + chunk_size]
        if len(chunk) < 2:
            continue

        best_char, best_score = None, -np.inf
        for ch in chars:
            try:
                score = models[ch].score(chunk)
                if score > best_score:
                    best_score = score
                    best_char = ch
            except:
                pass

        if best_char:
            predicted.append(best_char)

    return "".join(predicted)


# --- Run it ---
print("=" * 60)
print("Training GMM-HMM models per character...")
print(f"  n_components={2}, n_mix={2}, n_iter={5}, max_segments={MAX_SEGMENTS_PER_CHAR}")
print("=" * 60)

overall_start = time.time()
hmm_models = build_char_hmm_models(train_labels, train_feats)
overall_elapsed = time.time() - overall_start

print(f"\n{'=' * 60}")
print(f"Total training time: {overall_elapsed:.1f}s")
print(f"Trained {len(hmm_models)} character models")
print("=" * 60)

Estimated frames per character: 11
Average utterance length: 1285 frames, 114 chars
Training GMM-HMM models per character...
  n_components=2, n_mix=2, n_iter=5, max_segments=500
Step 1/3: Collecting frames per character...
  Processed 3400/3443 utterances
  Done. Found 150 unique characters.
  Characters: [' ', '!', '"', '%', '&', "'", '(', ')', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'x', 'y', '¥', '°', '²', 'Õ', 'õ', '،', '؛', '؟', 'ء', 'آ', 'ئ', 'ا', 'ب', 'ت', 'ث', 'ج', 'ح', 'خ', 'د', 'ذ', 'ر', 'ز', 'س', 'ش', 'ص', 'ض', 'ط', 'ظ', 'ع', 'غ', 'ـ', 'ف', 'ق', 'ل', 'م', 'ن', 'ه', 'و', 'ي', 'ً', 'َ', 'ُ', 'ِ', 'ٰ', 'ٺ', 'ٻ', 'ٽ', 'پ', 'ٿ', 'ڀ', 'ڃ', 'ڄ', 'چ', 'ڇ', 'ڈ', 'ڊ', 'ڌ', 'ڍ', 'ڏ', 'ڙ', 'ڦ', 'ک', 'ڪ', 'گ', 'ڱ', 'ڳ', '

Step 3B: Acoustic Model — SVM / Random Forest on MFCC statistics

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def utterance_features(feats):
    """
    Summarize variable-length MFCC sequence into fixed-size vector.
    Mean + std over time → (78,) vector.
    """
    return np.concatenate([feats.mean(axis=0), feats.std(axis=0)])

# Build character-level training data
# (Each utterance → one label: first character, or full transcription hash for demo)
X_train = np.array([utterance_features(f) for f in train_feats])
X_test  = np.array([utterance_features(f) for f in test_feats])

# For a real system, align features to phonemes first.
# Here we predict the first character as a sanity check demo.
y_train = [t[0] if t else " " for t in train_labels]
y_test  = [t[0] if t else " " for t in test_labels]

svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    SVC(kernel="rbf", C=10, gamma="scale"))
])

print("Training SVM...")
svm_pipeline.fit(X_train, y_train)
acc = svm_pipeline.score(X_test, y_test)
print(f"First-character accuracy (SVM): {acc * 100:.1f}%")

Training SVM...
First-character accuracy (SVM): 15.5%


Step 4: Language Model — n-gram (replaces Whisper's decoder)

In [ ]:
from collections import defaultdict
import math

def build_ngram_lm(transcriptions, n=3):
    """
    Character-level n-gram language model.
    Analogous to Whisper's autoregressive decoder prior.
    """
    ngram_counts = defaultdict(lambda: defaultdict(int))
    context_counts = defaultdict(int)

    for text in transcriptions:
        padded = " " * (n - 1) + text + " "
        for i in range(len(padded) - n + 1):
            context = padded[i:i + n - 1]
            next_ch = padded[i + n - 1]
            ngram_counts[context][next_ch] += 1
            context_counts[context] += 1

    return ngram_counts, context_counts

def ngram_score(text, ngram_counts, context_counts, n=3, alpha=0.1):
    """Compute log-prob of a string under the n-gram LM (with Laplace smoothing)."""
    padded = " " * (n - 1) + text
    log_prob = 0.0
    vocab = set(ch for counts in ngram_counts.values() for ch in counts)
    V = len(vocab)

    for i in range(n - 1, len(padded)):
        context = padded[i - n + 1:i]
        ch = padded[i]
        count = ngram_counts[context].get(ch, 0)
        total = context_counts[context]
        prob = (count + alpha) / (total + alpha * V + 1e-9)
        log_prob += math.log(prob + 1e-9)

    return log_prob

print("Building character 3-gram LM from training transcriptions...")
ngram_counts, context_counts = build_ngram_lm(train_labels, n=3)
print(f"LM built with {len(ngram_counts)} unique contexts")

Building character 3-gram LM from training transcriptions...
LM built with 2265 unique contexts


Step 5: Evaluation — WER & CER (same metrics as your Whisper code)

In [ ]:
import evaluate

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def evaluate_classical(features_list, references, models, sample_size=100):
    predictions, refs_clean = [], []

    for i, (feats, ref) in enumerate(zip(features_list[:sample_size],
                                         references[:sample_size])):
        pred = decode_with_hmm(feats, models)
        predictions.append(pred.lower())
        refs_clean.append(ref.lower())

        if i % 20 == 0:
            print(f"  {i}/{sample_size} | Ref: {ref[:30]!r:30} | Pred: {pred[:30]!r}")

    wer = wer_metric.compute(predictions=predictions, references=refs_clean)
    cer = cer_metric.compute(predictions=predictions, references=refs_clean)

    print(f"\n{'Model':<25} {'WER':>8} {'CER':>8}")
    print("-" * 43)
    print(f"{'GMM-HMM (classical)':<25} {wer*100:>7.2f}% {cer*100:>7.2f}%")
    return wer, cer

wer, cer = evaluate_classical(test_feats, test_labels, hmm_models)

  0/100 | Ref: "'هوسٽائل ماحول واري ڪورس' لاءِ" | Pred: 'yٿ......!ffPڱشڏڱKL“بBڌ,ً-ٽيLrO'


  20/100 | Ref: 'ڪجهہ فعلن ۽ شين جي وچ ۾ فرق ڪر' | Pred: 'AyyyyMyyHظeDٰڌA؟,Iن:/JIBrاظIIH'


  40/100 | Ref: 'ننڊ ۾ رنڊڪ توهان جي عام ننڊ جي' | Pred: 'AaAA—eAKنڌڌ(”OSaAAA RK(RRL”KK3'


  60/100 | Ref: 'ڪيوومو، 53 سال عمر، هن سال جي ' | Pred: ',y———A——.A——y—Aہruooوڌrrظُس/ڌڌ'


  80/100 | Ref: 'جڏهن تہ گوما مناسب حد تائين مح' | Pred: '2AAMڦ—AGگOہڌ:ABھٻiڄڌHRKBrڍڊشڍK'



Model                          WER      CER
-------------------------------------------
GMM-HMM (classical)        100.00%  102.02%


In [ ]:
print("=" * 70)
print("SAMPLE PREDICTIONS: GMM-HMM Classical Model")
print("=" * 70)

for i in range(10):
    feats = test_feats[i]
    ref   = test_labels[i]
    pred  = decode_with_hmm(feats, hmm_models)

    print(f"\nExample {i+1}")
    print(f"  Reference:  {ref}")
    print(f"  Predicted:  {pred}")
    print(f"  Match:      {'✓' if ref.lower() == pred.lower() else '✗'}")
    print("-" * 70)

SAMPLE PREDICTIONS: GMM-HMM Classical Model



Example 1
  Reference:  'هوسٽائل ماحول واري ڪورس' لاءِ انٽرنيٽ جي هڪ ڳولا ممڪن طور تي ڪنهن مقامي ڪمپني جو پتو مهيا ڪندي.
  Predicted:  yٿ......!ffPڱشڏڱKL“بBڌ,ً-ٽيLrOًوAA, ب:Bم'MنڀٰB؟¥ ن ؟,,F,y,KيMڌ/ڌE(بييڊ Kڌ Bڃ,!.Nyypyypyyی
  Match:      ✗
----------------------------------------------------------------------



Example 2
  Reference:  هن ڪٽس لاءِ ڪو انگ اکر اهو چئي سيٽ نه ڪيو ته، اهي چين جي اِقتصادي پيداوار جي بُنياد تي سيٽ ڪيا ويندا.
  Predicted:  M————Ep/ظڏس؟GTاGوووا—Aي(؟)6Rڌc(ڌBسT”Fڇ“Gا,ظاeيي)وoA———Tس.———AHڀMس%ڌrٰاسِٰٰڄڏ(4lT(ٰڌ/IڌFڇ“GڌrTثہڌ..—A———
  Match:      ✗
----------------------------------------------------------------------



Example 3
  Reference:  اوٽاوا ڪينيڊا جو من موهيندڙ، ٻن ٻولين وارو گادي جو هنڌ آهي ۽ هن ۾ آرٽ گيلريون ۽ عجائب گهرن جي هڪ قطار آهي جيڪو ڪينيڊا جي ماضي ۽ حال کي ظاهر ڪري ٿي.
  Predicted:  A.ًZZ.AAfffZ..HڍٿُBMٰ,ڌڌR,,rبڄڍuا.HڌڌڀTبڌ..A!..pبڌکٰٰuNrRM‘Nڱن+3ڌيطبHڱجMyECين,uه-ِ.ڏڱM)/ڌٰڌ,ي)يظڱٰڌ)ڍ://MڏMڌء ,AAB ڏBrr,ُيڌHcَ/1يڌڊnمٰظr)ظ)Rک.yDA.AZZ
  Match:      ✗
----------------------------------------------------------------------



Example 4
  Reference:  روم ۾ ڪيترين ئي جاين تي ماڻهن کي رسم ڏسڻ لاءِ ڪيترين ئي وڏين اسڪرينن وارين ٽيلي ويزنن کي نصب ڪيون ويو هو.
  Predicted:  AZyy.AZ...يRڱ4ڌlاٰBيڌبظ-؛Bبڊڱ(ی /cس(ظNڌHrmA!AAبڌيڌڌغ,ڌgڍrڌ(M—Z.yDMڄ)M,ڌ/,(ڌڌc اظuL)rrEyy..y.y.
  Match:      ✗
----------------------------------------------------------------------


...


Example 6
  Reference:  جيانڪارلو فسيشلا پنهنجي ڪار جو ڪنٽرول وڃائي ڇڏيو ۽ شروع ڪرڻ کان تمام جلد ريس ختم ڪري ڇڏي.
  Predicted:  AeeAeeApAAAAAeAee'–ض0l0AApھaI)sAAAAAAچس؟–ُiHK0ھ+cَڏغبcT/ُڏڏٰMڄڏڌ”MًyًFَبببDRiشuٺ8ظ/7A,0ظًRڏڌ/ڏ,0AApAAAAe
  Match:      ✗
----------------------------------------------------------------------



Example 7
  Reference:  تنهنڪري، امڪان اهو آهي تہ هن اشاري کي صرف ليبل جي طور تي شامل ڪيو ويو هو.
  Predicted:  C.B—..,ظڏGUUٿہrڻ4ئrr/)'U.,-h.ARڏووceaD,-)ث(ف؟ظڄDU,/ثفث,4T,Uحe,ڇ/Gڻخ(RRوو..BH)V——..e2-e.y—.Bڌ.e..
  Match:      ✗
----------------------------------------------------------------------



Example 8
  Reference:  هڪ مال گاڏي ۾، اهي بادشاهه ۽ راڻي جي خلاف خطرا ۽ ڌمڪيون ۽ نعرا هڻندڙ ماڻهن جي هجوم ۾ شامل ٿي پيرس ڏانهن واپس سفر ڪيو.
  Predicted:  D.!—AفقاڌڌظظG3نplڌيr, س,rULFڇ,%/صبE/ڇڌڃڇGoUغرظrR/ڌr-ج,ڇسFَ,DEڀصGTصmmي’Gسٰ؟,.یی.(RGسسسElVB————
  Match:      ✗
----------------------------------------------------------------------



Example 9
  Reference:  ميٽرو ۾ ٿيندڙ اعلان صرف ڪيٽلين ۾ ٿيندا آهن، پر اڻ رٿيل خللن جو اعلان خودڪار نظام جي ذريعي مخلتف زبانن ۾ ٿيندا آهن جنهن ۾ اندلسي، فرانسيسي، عربي ۽ جاپاني شامل آهن.
  Predicted:  AAAApOAAAA8ڏٰڌمBڌyAAA8ڌnڀسشu(AAyAAڏ3T”uڊیٰ0)cAyAAڏبڀA—yA1ڌrڀصڌAAyyyEAAڳڀڊ)ش)AyyبصnڌغڏڌTڱڌظagڀHاHيMبuHHڌهڌGr:Myy.G0ڻ0ڌ—yyyAyyyAAغڀظ0صTٿ—yAشڱDصصص”AyابڀڌڻAyصا8غڱ؟ِrH:Myy—y——————
  Match:      ✗
----------------------------------------------------------------------



Example 10
  Reference:  ايٽلانٽڪ سامونڊي طوفان جي موسم جو ڏهون نالي طوفان، سب ٽراپيڪل طوفان جيري، جيڪو اڄ ايٽلانٽڪ سمنڊ ۾ تيار ٿيو.
  Predicted:  A——AAA——,صrڍ0)——————,بڊڌoڱoٰڱسaهڊہgصڌUڻSAA.ش؟ڄهڊHوuٰظا.—سفNڌTAeظٻ(شrٰہ؛.س(ڄاد)ڻددڱٰٰہG0ڊrظڻgڊڊf3ٰVV————
  Match:      ✗
----------------------------------------------------------------------


In [ ]:
print(f"WER: {wer*100:.2f}%")
print(f"CER: {cer*100:.2f}%")

WER: 100.00%
CER: 102.02%
